# Batch norm derivation

$$
y = \frac{x - \mathrm{E}[x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} * \gamma + \beta
$$

Best pen and paper source is: 

Kevin Zakka's blog post: "Deriving the Gradient for the Backward Pass of Batch Normalization".

4. The Original Source: Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift

3. The Math Derivation: Deriving the Gradient for the Backward Pass of Batch Normalization
By: Frederik Kratzert (Blog Post / Markdown Guide)

https://www.youtube.com/watch?v=Y23QgQGAGJQ


Matrix calculus you need for DL. 
https://arxiv.org/pdf/1802.01528

1. It explicitly focuses on the Vector-Jacobian Product (VJP)
Instead of forcing you to memorize giant Jacobian matrices, Howard and Parr introduce the math from a "row vector times Jacobian" perspective. They show you how to trace a vector of downstream gradients through operations without ever materializing a giant, sparse, memory-wasting matrix in your code.

Strategy:
- decompose all the above into individual vjp


In [279]:
import torch
from torch.nn import BatchNorm1d 

# ensure determinism

In [280]:
torch.manual_seed(0)
num_feats = 5
a = BatchNorm1d(num_features=num_feats)
examples=3
data = torch.rand((examples, num_feats))
print(a(data))
print(data)

tensor([[ 0.0273,  1.3757, -0.6222, -0.8304, -0.6633],
        [ 1.2104, -0.4049,  1.4109,  1.4062,  1.4130],
        [-1.2377, -0.9707, -0.7887, -0.5758, -0.7498]],
       grad_fn=<NativeBatchNormBackward0>)
tensor([[0.4963, 0.7682, 0.0885, 0.1320, 0.3074],
        [0.6341, 0.4901, 0.8964, 0.4556, 0.6323],
        [0.3489, 0.4017, 0.0223, 0.1689, 0.2939]])


In [4]:
list(a.parameters())

list(a.named_parameters())

[('weight',
  Parameter containing:
  tensor([1., 1., 1., 1., 1.], requires_grad=True)),
 ('bias',
  Parameter containing:
  tensor([0., 0., 0., 0., 0.], requires_grad=True))]

In [ ]:
# want to output a class:

In [276]:
import numpy as np

X = data.detach().cpu().numpy()
epsilon = 1e-5

gamma = np.ones(num_feats)
beta = np.zeros(num_feats)

# num_feats = 5, num_examples = 3
# X = (3, 5),

y = (X - np.mean(X, axis=0)) / np.sqrt(np.var(X, axis=0) + epsilon)
y

array([[ 0.02730129,  1.3756715 , -0.6222197 , -0.83038163, -0.66327256],
       [ 1.2104148 , -0.40493888,  1.4108994 ,  1.4062171 ,  1.4130424 ],
       [-1.2377157 , -0.97073305, -0.7886797 , -0.5758355 , -0.7497701 ]],
      dtype=float32)

In [286]:
X


array([[0.4962566 , 0.7682218 , 0.08847743, 0.13203049, 0.30742282],
       [0.6340787 , 0.4900934 , 0.89644474, 0.45562798, 0.6323063 ],
       [0.34889346, 0.4017173 , 0.02232575, 0.16885895, 0.29388845]],
      dtype=float32)

# Now write in terms of ops

0. Summation / scaling gate (mean) - 1/n sum over batch
1. Subtraction 
2. Power 
3. Summation / scaling ()
4. Denominator
5. inverse
6. mm


# Example

X is (3, 5) i.e. 3 examples and 5 features

mean should be size 5 (average over each feature)

In [100]:
# def sum_scale(scale, X):
#     # collapses the batch dim
#     return scale * np.sum(X, axis=0)


def sub(a, b):
    return a - b


def add(a, b):
    return a + b


def power(a, pow):
    # return np.array([j ** pow for i in a for j in i])
    return a**pow


def mm(a, b):
    return a * b


def hadamard(a, b):
    return a * b


# dldp is upstream
# def vjp_hadamard(a, b, dldp):
#     # chain rule p = a * b
#     # dlda = dldp * dpda
#     # dldb = dldp * dpdb

#     return


## initialisation of parameters (we are training gamma and bias)
num_feats = X.shape[1]
num_examples = X.shape[0]
gamma = np.ones(num_feats)  # num feats
bias = np.zeros(num_feats)

## Summation
sum0 = np.sum(X, axis=0)  # reduction # collapse: vector of size 5 i.
mean = (1.0 / len(X)) * sum0  # compute mean
meanb = np.broadcast_to(mean, X.shape)
num = sub(X, meanb)  # size (3,5) # broadcasting (possible extra op)

# Denominator
## Variance
b = pow(num, 2)  # size (3, 5) # applied elementwise
sum0_ = np.sum(b, axis=0)  # collapse: vector of size 5 i.e. feats
var = (1.0 / len(X)) * sum0_  # (5,)
eps_b = np.broadcast_to(epsilon, var.shape)  # Broadcast: () -> (5,)
var_eps = add(var, eps_b)  # Elementwise: (5,) + scalar
denom_vec = power(var_eps, -0.5)  # Pure Power: (5,)
denom_b = np.broadcast_to(denom_vec, X.shape)  # Broadcast:  (5,) -> (3, 5)

# --- OUTPUT ---
bn = hadamard(num, denom_b)  # (3, 5) * (3, 5)
gamma_ = np.broadcast_to(gamma, X.shape)  # (5,) -> (3, 5)
bn_gamma = hadamard(bn, gamma_)  # (3 * 5) * (1 * 5)

bias_ = np.broadcast_to(bias, bn_gamma.shape)
bn_gamma_bias = add(bn_gamma, bias)
bn

array([[ 0.02730103,  1.37567159, -0.62221974, -0.83038162, -0.66327255],
       [ 1.21041441, -0.4049389 ,  1.41089934,  1.40621713,  1.41304242],
       [-1.23771595, -0.97073307, -0.78867975, -0.57583551, -0.74977006]])

In [62]:

a = np.array(([1,2,3], [4,5,6], [7,8,9]))

pow(a,2)
# def pow(a, 2)
# a

array([[ 1,  4,  9],
       [16, 25, 36],
       [49, 64, 81]])

In [54]:
assert np.allclose(bn, y)
bn

array([[ 0.02730103,  1.37567159, -0.62221974, -0.83038162, -0.66327255],
       [ 1.21041441, -0.4049389 ,  1.41089934,  1.40621713,  1.41304242],
       [-1.23771595, -0.97073307, -0.78867975, -0.57583551, -0.74977006]])

# Now do vjps

$$
\newcommand{\pd}[2]{\frac{\partial #1}{\partial #2}}
$$

## Vjp of broadcast and reduction:
To make this easier I have explicitly used broadcast and reduction in the forward chain. I also broke the sum scale into two separate ops. Scaling by a constant is trivial.
- there are some downsides to this that I will discuss later
- benefits
However we have to put in control flow checks for different dimensions of a tensor. 

some notes: 
- add_scalar is trivial 

We now need to implement these vjps in reverse. 
### vjp of broadcast_to()
The vjp of the broadcast

Going from b_j to B_{ij}

B_ij = fn(b_j) therefore we differentiated L w.r.t B_ij

$$
\begin{aligned}
B_{ij} &= b_j; \quad \text{or} \\
b_j &= \sum_i B_{ij} \\

\text{vjp of broadcast:} \quad \pd{L}{B_{ij}} = \pd{L}{b_k} \pd{b_{k}}{B_{ij}} &= \pd{L}{b_k}\sum_l \pd{B_{lk}}{B_{ij}} = \pd{L}{b_k}\sum_l \delta_{il} \delta_{jk} \\
\quad \quad \text{i.e. the r.h.s sum is non-zero only when l = k } \\

\text{vjp of reduction:} \quad \pd{L}{b_k} = \pd{L}{B_{ij}} \pd{B_{ij}}{b_k} &= \pd{L}{B_{ij}}\delta_{kj} = \pd{L}{B_{ik}}
\end{aligned}
$$

So we see that the vjp of a broadcast is a reduction, and the vjp of a sum is a broadcast. 


### vjp of hadamard()
$$
\begin{aligned}
p &= a \odot b\\
\pd{L}{a_{ij}} &= \pd{L}{p_{kl}} \pd{p_{kl}}{a_{ij}} \\
\pd{L}{b_{ij}} &= \pd{L}{p_{kl}} \pd{p_{kl}}{b_{ij}} \\
p_{ij} &= a_{ij} b_{ij}\\
\pd{p_{ij}}{a_{kl}} = \pd{(a_{ij} b_{ij})}{a_{kl}} = b_{ij} \pd{a_{ij}}{a_{kl}} &= b_{ij}\delta_{ki}\delta_{lj} \\
 \pd{p_{kl}}{a_{ij}}  &= b_{kl}\delta_{ik}\delta_{jl} \\
\pd{L}{a_{ij}} &= \pd{L}{p_{kl}} b_{kl} \delta_{ik}\delta_{jl} = \pd{L}{p_{ij}} b_{ij}\\
\end{aligned}
$$

ops: dL/dB 

$$
B = B(\gamma, \beta)
$$

$$
s = \sum_b X_{bj}
$$

vjp
$$
\frac{\partial s }{\partial X_{lm}} = \sum_b \frac{\partial X}{\partial}

$$





In [ ]:
def vjp_add(a, b, dldp):
    dlda = dldp
    dldb = dldp
    return dlda, dldb


def vjp_sub(a, b, dldp):
    dlda = dldp
    dldb = dldp
    return dlda, -dldb


def vjp_mm(a, b, dldp):
    dlda = dldp @ b
    dldb = dldp.T @ a
    return dlda, dldb


# p = scale * a_{ij}
# dlda = dldp *
def vjp_scale(scale, dldp):
    # dlda = dldp * dpda
    dlda = dldp * scale
    return dlda


def vjp_hadamard(a, b, dldp):
    # print(dldp.shape, b.shape, a.shape)
    dlda = dldp * b
    dldb = dldp * a
    return dlda, dldb


def vjp_power(a, alpha, dldp):
    dlda = (
        alpha * dldp * pow(a, alpha - 1.0)
    )  # local grad or jacob is alpha * pow(a, alpha-1)
    return dlda


def vjp_reduction(A, shape):
    return np.broadcast_to(A, shape)


def vjp_broadcast(A):
    return np.sum(A, axis=0)


In [257]:
# assert np.allclose(bn, y)

# Attempt forward back

In [258]:
def test(num_feats=5, num_examples=3):
    torch.manual_seed(0)
    bn = BatchNorm1d(num_features=num_feats)
    data = torch.rand((examples, num_feats))
    print(bn(data).detach().cpu().numpy())

    assert np.allclose(bn(data).detach().cpu().numpy(), bn_manual(X))


def init():
    X = torch.rand((examples, num_feats))


def bn_manual(X, gamma=np.ones((X.shape[1],)), eps=1e-5, bias=np.zeros((X.shape[1],))):
    ## Summation
    sum0 = np.sum(X, axis=0)  # reduction # collapse: vector of size 5 i.
    mean = (1.0 / len(X)) * sum0  # compute mean
    meanb = np.broadcast_to(mean, X.shape)
    num = sub(X, meanb)  # size (3,5) # broadcasting (possible extra op)

    # Denominator
    ## Variance
    b = pow(num, 2)  # size (3, 5) # applied elementwise # num
    sum0_ = np.sum(b, axis=0)  # collapse: vector of size 5 i.e. feats
    var = (1.0 / len(X)) * sum0_  # (5,)
    eps_b = np.broadcast_to(eps, var.shape)  # Broadcast: () -> (5,)
    var_eps = add(var, eps_b)  # Elementwise: (5,) + scalar
    denom_vec = power(var_eps, -0.5)  # Pure Power: (5,)
    denom_b = np.broadcast_to(denom_vec, X.shape)  # Broadcast:  (5,) -> (3, 5)

    # --- OUTPUT ---
    bn = hadamard(num, denom_b)  # (3, 5) * (3, 5) # num
    gamma_ = np.broadcast_to(gamma, X.shape)  # (5,) -> (3, 5)
    bn_gamma = hadamard(bn, gamma_)  # (3 * 5) * (1 * 5)

    bias_ = np.broadcast_to(bias, bn_gamma.shape)
    bn_gamma_bias = add(bn_gamma, bias_)

    return bn_gamma_bias


# let's start by setting the upstream loss dldbn = some random array
def bn_backward(dldp):  # we want dldbias, dldgamma and dldx?
    # p = bn(X,gamma, bias)
    # vjps:
    # dldbias = dldp * dpdbias
    # dldgamma = dldp * dgamma
    # dldX = dldp * dpdX

    # c.f. y = linear(X, theta)
    # dl/dy = dl
    #

    # OR I can begin from my last ops:

    # i.e. dlbn_gamma = dlbn_gamma_bias * dbn_gamma_biasdbias
    # dldngamma = dLd

    # we can kick this off by setting dldp to anything. remember that vjp

    # bn_gamma_bias = add(bn_gamma, bias_)
    dldbn_gamma, dlbias_ = vjp_add(bn_gamma, bias_, dldp)  # we return dlbn_gamma

    # bias_ = np.broadcast_to(bias, bn_gamma.shape)
    dldbias = vjp_broadcast(dlbias_)  # this reduces

    # bn_gamma = hadamard(bn, gamma_)  # (3 * 5) * (1 * 5)
    dldbn, dldgamma_ = vjp_hadamard(
        bn, gamma_, dldbn_gamma
    )  # this was previously wrong because I used dldbias

    # Collapse gamma_ (3, 5) -> gamma (5,)
    # gamma_ = np.broadcast_to(gamma, X.shape)  # (5,) -> (3, 5)
    dldgamma = vjp_broadcast(dldgamma_)

    # bn = hadamard(num, denom_b)  # (3, 5) * (3, 5)
    dldnum, dldenom_b = vjp_hadamard(num, denom_b, dldbn)

    # denom_b = np.broadcast_to(denom_vec, X.shape)  # Broadcast:  (5,) -> (3, 5)
    dldnom_vec = vjp_broadcast(dldenom_b)

    # denom_vec = power(var_eps, -0.5)  # Pure Power: (5,)
    dldvar_eps = vjp_power(var_eps, -0.5, dldnom_vec)

    # var_eps = add(var, eps_b)  # Elementwise: (5,) + scalar
    dldvar, dldeps_b = vjp_add(var, eps_b, dldvar_eps)

    # eps_b = np.broadcast_to(eps, var.shape)  # Broadcast: () -> (5,)
    # var = (1.0 / len(X)) * sum0_  # (5,)
    dldsum0_ = vjp_scale(1.0 / len(X), dldvar)

    # sum0_ = np.sum(b, axis=0)  # collapse: vector of size 5 i.e. feats
    dldb = vjp_reduction(dldsum0_, b.shape)

    # b = pow(num, 2)  # size (3, 5) # applied elementwise
    dldnum += vjp_power(num, 2, dldb)

    # num = sub(X, meanb)  # size (3,5) # broadcasting (possible extra op)
    dldX, dldmeanb = vjp_sub(
        X, meanb, dldnum
    )  # size (3,5) # broadcasting (possible extra op)

    # meanb = np.broadcast_to(mean, X.shape)
    dldmean = vjp_broadcast(dldmeanb)

    # mean = (1.0 / len(X)) * sum0  # compute mean
    dldsum0 = vjp_scale(1.0 / len(X), dldmean)

    # sum0 = np.sum(X, axis=0)  # reduction # collapse: vector of size 5 i.
    dldX += vjp_reduction(dldsum0, X.shape)

    return dldX, dldgamma, dldbias
    # we will get a branching node trap - basically when two




# typically Linear, Relu, dropout.


$$
y = \frac{x - \mathrm{E}[x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} * \gamma + \beta
$$

In [292]:
np.random.seed(43)
# Set X_new
X_new = np.random.randn(*X.shape).astype(np.float32)

# Set non-zero upstream gradient
dldp_np = np.random.randn(*X.shape).astype(np.float32)

print(X_new)

print(dldp_np)

[[ 0.25739992 -0.9084814  -0.3785031  -0.5349156   0.85807335]
 [-0.41300997  0.49818859  2.0101993   1.2628615  -0.43921486]
 [-0.3464379   0.45531967 -1.6686627  -0.8620855   0.49291086]]
[[-0.1243134   1.9351363  -0.61844265 -1.046839   -0.88961756]
 [ 0.01404054 -0.1608297   2.2303596  -0.3991157   0.05444456]
 [ 0.8841818  -0.10798056  0.55560696  0.39490664  0.83720505]]


In [302]:
def fwd_backwd(X, dldp, gamma=np.ones((X.shape[1],), dtype=np.float32), eps=1e-5, bias=np.zeros((X.shape[1],), np.float32)):
    ## Summation
    sum0 = np.sum(X, axis=0)  # reduction # collapse: vector of size 5 i.
    mean = (1.0 / len(X)) * sum0  # compute mean
    meanb = np.broadcast_to(mean, X.shape)
    num = sub(X, meanb)  # size (3,5) # broadcasting (possible extra op)

    # Denominator
    ## Variance
    b = pow(num, 2.)  # size (3, 5) # applied elementwise # num
    sum0_ = np.sum(b, axis=0)  # collapse: vector of size 5 i.e. feats
    var = (1.0 / len(X)) * sum0_  # (5,)
    eps_b = np.broadcast_to(eps, var.shape)  # Broadcast: () -> (5,)
    var_eps = add(var, eps_b)  # Elementwise: (5,) + scalar
    denom_vec = power(var_eps, -0.5)  # Pure Power: (5,)
    denom_b = np.broadcast_to(denom_vec, X.shape)  # Broadcast:  (5,) -> (3, 5)

    # --- OUTPUT ---
    bn = hadamard(num, denom_b)  # (3, 5) * (3, 5) # num
    gamma_ = np.broadcast_to(gamma, X.shape)  # (5,) -> (3, 5)
    bn_gamma = hadamard(bn, gamma_)  # (3 * 5) * (1 * 5)
    bias_ = np.broadcast_to(bias, bn_gamma.shape)
    bn_gamma_bias = add(bn_gamma, bias_)

    fwd = bn_gamma_bias

    ## BACKWARD:
    # bn_gamma_bias = add(bn_gamma, bias_)
    dldbn_gamma, dlbias_ = vjp_add(bn_gamma, bias_, dldp)  # we return dlbn_gamma

    # bias_ = np.broadcast_to(bias, bn_gamma.shape)
    dldbias = vjp_broadcast(dlbias_)  # this reduces

    # bn_gamma = hadamard(bn, gamma_)  # (3 * 5) * (1 * 5)
    dldbn, dldgamma_ = vjp_hadamard(bn, gamma_, dldbn_gamma)

    print("dldbn_gamma ", dldbn_gamma)
    # Collapse gamma_ (3, 5) -> gamma (5,)
    # gamma_ = np.broadcast_to(gamma, X.shape)  # (5,) -> (3, 5)
    dldgamma = vjp_broadcast(dldgamma_)
    print("dldgamma reduced", dldgamma)

    ## DO NOT CARE BELOW THIS
    # bn = hadamard(num, denom_b)  # (3, 5) * (3, 5)
    dldnum, dldenom_b = vjp_hadamard(num, denom_b, dldbn)

    ## Variance
    # denom_b = np.broadcast_to(denom_vec, X.shape)  # Broadcast:  (5,) -> (3, 5)
    dldnom_vec = vjp_broadcast(dldenom_b)

    # denom_vec = power(var_eps, -0.5)  # Pure Power: (5,)
    dldvar_eps = vjp_power(var_eps, -0.5, dldnom_vec)

    # var_eps = add(var, eps_b)  # Elementwise: (5,) + scalar
    dldvar, dldeps_b = vjp_add(var, eps_b, dldvar_eps)

    # eps_b = np.broadcast_to(eps, var.shape)  # Broadcast: () -> (5,)
    # var = (1.0 / len(X)) * sum0_  # (5,)
    dldsum0_ = vjp_scale(1.0 / len(X), dldvar)

    # sum0_ = np.sum(b, axis=0)  # collapse: vector of size 5 i.e. feats
    dldb = vjp_reduction(dldsum0_, b.shape)

    ## Numerator
    # b = pow(num, 2)  # size (3, 5) # applied elementwise
    dldnum += vjp_power(num, 2.0, dldb)

    # num = sub(X, meanb)  # size (3,5) # broadcasting (possible extra op)
    dldX, dldmeanb = vjp_sub(
        X, meanb, dldnum
    )  # size (3,5) # broadcasting (possible extra op)

    # meanb = np.broadcast_to(mean, X.shape)
    dldmean = vjp_broadcast(dldmeanb)

    # mean = (1.0 / len(X)) * sum0  # compute mean
    dldsum0 = vjp_scale(1.0 / len(X), dldmean)

    # sum0 = np.sum(X, axis=0)  # reduction # collapse: vector of size 5 i.
    dldX += vjp_reduction(dldsum0, X.shape)

    return fwd, dldX, dldgamma, dldbias

# print(X)
fwd, dldX, dldgamma, dldbias = fwd_backwd(X_new, dldp_np)

print(fwd)
# print(dldgamma, '\n', dldbias)
# np.pow(a,-1)

dldbn_gamma  [[-0.1243134   1.9351363  -0.61844265 -1.046839   -0.88961756]
 [ 0.01404054 -0.1608297   2.2303596  -0.3991157   0.05444456]
 [ 0.8841818  -0.10798056  0.55560696  0.39490664  0.83720505]]
dldgamma reduced [-0.71156326 -2.92742289  2.50460308 -0.35484597 -0.68693415]
[[ 1.40838141 -1.4136894  -0.2402685  -0.52473073  1.01451043]
 [-0.81456032  0.7396568   1.32707119  1.39967626 -1.36049968]
 [-0.59382099  0.67403259 -1.08680266 -0.87494549  0.34598926]]


In [300]:
from torch.nn import BatchNorm1d
from torch import tensor

import torch.nn as nn

model = BatchNorm1d(num_features=5, dtype=torch.float32)
model.zero_grad()
fwd_pt = model(tensor(X_new, dtype=torch.float32))

fwd_pt.backward(tensor(dldp_np))
params = model.parameters()

params_l = list(params)

print(fwd_pt)
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.grad)

# print(model.weight.grad)
# print(model.bias.grad)

tensor([[ 1.4084, -1.4137, -0.2403, -0.5247,  1.0145],
        [-0.8146,  0.7397,  1.3271,  1.3997, -1.3605],
        [-0.5938,  0.6740, -1.0868, -0.8749,  0.3460]],
       grad_fn=<NativeBatchNormBackward0>)
weight tensor([-0.7116, -2.9274,  2.5046, -0.3548, -0.6869])
bias tensor([ 7.7391e-01,  1.6663e+00,  2.1675e+00, -1.0510e+00,  2.0321e-03])


In [254]:
assert np.allclose(fwd_pt.detach().cpu().numpy(), fwd)

In [262]:
-5.11667266e-07 / -2.

2.55833633e-07

In [255]:
model = nn.Linear(5,3)
x_input = torch.rand((1,5), requires_grad=True)
dldp = torch.rand(1,3)
out = model(x_input)
out.backward(dldp)
out

dldX_pytorch = x_input.grad.numpy()
dldW_pytorch = model.weight.grad.numpy()
dldb_pytorch = model.bias.grad.numpy()

print(dldX_pytorch, dldW_pytorch, dldb_pytorch)

[[ 0.48777026  0.0327835   0.06416191  0.30801207 -0.2832333 ]] [[0.03421064 0.79688144 0.98689497 0.5964659  0.4630348 ]
 [0.00987258 0.22996566 0.28480014 0.17212932 0.13362353]
 [0.01585871 0.36940286 0.45748562 0.27649808 0.2146447 ]] [0.9921749  0.28632385 0.45993322]


# Custom BatchNorm

In [92]:
class CustomBatchNorm(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # eps, gamma, beta
        
        pass

    def forward(self, X):
        pass

    def backward(self, upstream):
        pass

In [ ]:
# how do we actually test the above? 


# Other ops?
- Broadcasting might be an option to be absolutely explicit - we have to do the opposite for vjp. 